# Extracting Data from separate cases to one json case file, while filtereing only relevent metadata

In [8]:
import json
import os
from datetime import datetime

def extract_cases(source_file, target_file):
    """
    Extracts filtered case data from a JSON input file and saves to target file,
    keeping ALL decisions (each with its adoption date) for every case.
    Appends or updates cases if target exists.
    """
    # --- Load source JSON ---
    with open(source_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # --- Load or initialize target JSON ---
    if os.path.exists(target_file):
        with open(target_file, "r", encoding="utf-8") as f:
            try:
                existing_cases = json.load(f)
            except json.JSONDecodeError:
                existing_cases = {}
    else:
        existing_cases = {}

    def first_or_empty(value):
        """Return first element if list, otherwise value or empty string."""
        if isinstance(value, list):
            return value[0] if value else ""
        return value or ""

    def parse_json_str(value):
        """Safely parse a JSON string like '{"code":"x","label":"y"}'."""
        if isinstance(value, str):
            try:
                return json.loads(value)
            except json.JSONDecodeError:
                return {}
        return value if isinstance(value, dict) else {}

    def parse_date(date_str):
        """Parse a date string to datetime for sorting (fallback to None)."""
        if not date_str:
            return None
        try:
            dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
            return dt.replace(tzinfo=None)   # normalize -> naive, so sorting never mixes
        except Exception:
            return None


    # --- Process each case ---
    for case_id, case_data in data.items():
        metadata = case_data.get("metadata", {})
        decisions = case_data.get("decisions", [])

        # --- Keep ALL decisions with their dates (newest first) ---
        all_decisions = []
        for decision in decisions:
            dec_meta = decision.get("metadata", {})
            adoption_date = first_or_empty(dec_meta.get("decisionAdoptionDate", []))

            decision_label = ""
            decision_types = dec_meta.get("decisionTypes")
            if isinstance(decision_types, list) and decision_types:
                dec_obj = parse_json_str(decision_types[-1])
                decision_label = dec_obj.get("label", "")

            all_decisions.append({
                "decisionDate": adoption_date,
                "decisionLabel": decision_label,
            })

        all_decisions.sort(
            key=lambda d: parse_date(d["decisionDate"]) or datetime.min,
            reverse=True,
        )

        # --- Handle case sectors ---
        sector_code = sector_label = ""
        case_sectors = metadata.get("caseSectors", [])
        if isinstance(case_sectors, list) and case_sectors:
            sec_item = parse_json_str(case_sectors[0])
            sector_code = sec_item.get("code", "")
            sector_label = sec_item.get("label", "")

        # --- Handle caseLegalBasis (always list for labels) ---
        legal_basis_list = metadata.get("caseLegalBasis", [])
        case_legal_basis_code = ""
        case_legal_basis_label = []

        if isinstance(legal_basis_list, list) and legal_basis_list:
            for lb_entry in legal_basis_list:
                lb_obj = parse_json_str(lb_entry)
                if lb_obj:
                    case_legal_basis_code = lb_obj.get("code", case_legal_basis_code)
                    label = lb_obj.get("label", "")
                    if isinstance(label, str) and label:
                        parts = [p.strip() for p in label.split("+")]
                        case_legal_basis_label.extend(parts)

        # --- Handle caseCompanies (always list, split by "/") ---
        companies_raw = metadata.get("caseCompanies", [])
        companies = []

        if isinstance(companies_raw, list) and companies_raw:
            for c in companies_raw:
                if isinstance(c, str):
                    parts = [p.strip() for p in c.split("/") if p.strip()]
                    companies.extend(parts)
        elif isinstance(companies_raw, str):
            companies = [p.strip() for p in companies_raw.split("/") if p.strip()]

        # --- Flatten and build output ---
        filtered_case = {
            "caseInstrument": first_or_empty(metadata.get("caseInstrument")),
            "caseNumber": first_or_empty(metadata.get("caseNumberPart")),
            "caseTitle": first_or_empty(metadata.get("caseTitle")),
            "caseSectorsCode": sector_code,
            "caseSectorLabel": sector_label,
            "caseCompanies": companies,
            "caseLegalBasisCode": case_legal_basis_code,
            "caseLegalBasisLabel": case_legal_basis_label,
            "caseLastDecisionDate": first_or_empty(metadata.get("caseLastDecisionDate")),
            "caseInitiationDate": first_or_empty(metadata.get("caseInitiationDate")),
            "decisions": all_decisions,
        }

        existing_cases[case_id] = filtered_case

    # --- Save result ---
    with open(target_file, "w", encoding="utf-8") as f:
        json.dump(existing_cases, f, ensure_ascii=False, indent=2)

    print(f"✅ Extracted {len(existing_cases)} cases and saved to '{target_file}' (all decisions kept).")


# Fills all Merger cases with a legal basis = Art. 105

In [9]:
def fill_merger_legal_basis(json_file, fill_value="Art. 105"):
    """
    Update all cases with caseInstrument == 'Merger' and null/empty caseLegalBasisLabel
    to a specified fill_value.
    """
    import json
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    count = 0
    for case in data.values():
        instrument = case.get("caseInstrument", "")
        basis_labels = case.get("caseLegalBasisLabel")
        is_basis_null = (
            basis_labels is None
            or (isinstance(basis_labels, list) and not basis_labels)
            or (isinstance(basis_labels, str) and (basis_labels.strip() == "" or basis_labels.strip().lower() == "null"))
        )
        if instrument == "Merger" and is_basis_null:
            case["caseLegalBasisLabel"] = [fill_value]
            count += 1

    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"✅ Filled {count} 'Merger' cases with missing caseLegalBasisLabel!")



In [10]:
extract_cases("Data/case-data-M.json", "cases.json")

fill_merger_legal_basis("cases.json")

extract_cases("Data/case-data-AT.json", "cases.json")

✅ Extracted 10967 cases and saved to 'cases.json' (all decisions kept).
✅ Filled 10218 'Merger' cases with missing caseLegalBasisLabel!
✅ Extracted 10967 cases and saved to 'cases.json' (all decisions kept).


In [11]:
import json

def starts_with_art(label):
    return isinstance(label, str) and label.strip().startswith("Art")

def keep_case(case):
    labels = case.get("caseLegalBasisLabel", [])
    if isinstance(labels, str):
        labels = [labels]
    if not isinstance(labels, list):
        return False
    # Keep only if at least one legal basis starts with "Art".
    # Empty/missing legal basis -> any() is False -> dropped.
    return any(starts_with_art(lbl) for lbl in labels)

# --- Load cases.json into memory ---
with open("cases.json", "r", encoding="utf-8") as f:
    all_cases = json.load(f)

# --- Filter, keeping result in memory for later use ---
filtered_cases = {
    case_id: case
    for case_id, case in all_cases.items()
    if keep_case(case)
}

dropped = len(all_cases) - len(filtered_cases)
print(f"Loaded {len(all_cases)} cases")
print(f"Kept    {len(filtered_cases)} cases (legal basis starts with 'Art')")
print(f"Dropped {dropped} cases (no legal basis starting with 'Art')")

Loaded 10967 cases
Kept    10919 cases (legal basis starts with 'Art')
Dropped 48 cases (no legal basis starting with 'Art')


In [5]:
def list_unique_legal_basis(data):
    legal_bases = set()
    for case in data.values():
        labels = case.get("caseLegalBasisLabel", [])
        if isinstance(labels, str):  # Handle string accidentally present
            labels = [labels]
        for label in labels:
            if isinstance(label, str) and label.strip():
                legal_bases.add(label.strip())

    print(f"Total unique legal basis: {len(legal_bases)}")
    for b in sorted(legal_bases):
        print("-", b)

# Usage example:
list_unique_legal_basis(all_cases)

Total unique legal basis: 19
- Art 105 TFEU (Ex 85 EC)
- Art 106 TFEU (Ex 86 EC)
- Art 23(1)e Regulation 2003/1
- Art 258 TFEU (Ex 226 EC)
- Art 65 ECSC Treaty
- Art. 101
- Art. 101 TFEU
- Art. 102
- Art. 102 TFEU
- Art. 105
- Art. 105 TFEU
- Art. 106
- Art. 37
- Art. 4 TFEU
- Art. 53
- Art. 53 EEA
- Art. 54
- Art. 54 EEA
- Art. 7 Reg.2003/1228


In [6]:
# to be edited
# - Art 105 TFEU (Ex 85 EC) to Art. 105
# - Art 106 TFEU (Ex 86 EC) to Art. 106
# - Art 23(1)e Regulation 2003/1 to Art. 23
# - Art 258 TFEU (Ex 226 EC) to Art. 258
# - Art 65 ECSC Treaty to Art. 65
# - Art. 101 TFEU to Art. 101
# - Art. 102 TFEU to Art. 102
# - Art. 105 TFEU to Art. 105
# - Art. 4 TFEU to Art. 4
# - Art. 53 EEA to Art. 53
# - Art. 54 EEA to Art. 54
# - Art. 7 Reg.2003/1228 to Art. 7

# Normalizes legal basis variations

In [12]:
# Mapping of patterns to normalized values
mapping = {
    "Art 105 TFEU (Ex 85 EC)": "Art. 105",
    "Art 106 TFEU (Ex 86 EC)": "Art. 106",
    "Art 23(1)e Regulation 2003/1": "Art. 23",
    "Art 258 TFEU (Ex 226 EC)": "Art. 258",
    "Art 65 ECSC Treaty": "Art. 65",
    "Art. 101 TFEU": "Art. 101",
    "Art. 102 TFEU": "Art. 102",
    "Art. 105 TFEU": "Art. 105",
    "Art. 4 TFEU": "Art. 4",
    "Art. 53 EEA": "Art. 53",
    "Art. 54 EEA": "Art. 54",
    "Art. 7 Reg.2003/1228": "Art. 7",
}

def normalize_legal_basis(data, mapping):
    """
    Normalize `caseLegalBasisLabel` values in the in-memory cases dict, IN PLACE,
    according to the mapping dict. Returns the same dict for convenience.
    """
    for case_id, case_data in data.items():
        if "caseLegalBasisLabel" in case_data:
            labels = case_data["caseLegalBasisLabel"]
            # Defensive: sometimes might be string/null, not list
            if isinstance(labels, str):
                labels = [labels]
            case_data["caseLegalBasisLabel"] = [
                mapping.get(label.strip(), label.strip())
                for label in labels if isinstance(label, str) and label.strip()
            ]
    return data

In [13]:
normalize_legal_basis(filtered_cases, mapping)
print("✅ Legal basis normalization complete.")

✅ Legal basis normalization complete.


# Cleaning and parsing caseSectorLabel based on the NACE 

In [16]:
import re

# NACE section names
NACE_SECTIONS = {
    "A": "AGRICULTURE, FORESTRY AND FISHING",
    "B": "MINING AND QUARRYING",
    "C": "MANUFACTURING",
    "D": "ELECTRICITY, GAS, STEAM AND AIR CONDITIONING SUPPLY",
    "E": "WATER SUPPLY; SEWERAGE, WASTE MANAGEMENT AND REMEDIATION ACTIVITIES",
    "F": "CONSTRUCTION",
    "G": "WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES",
    "H": "TRANSPORTATION AND STORAGE",
    "I": "ACCOMMODATION AND FOOD SERVICE ACTIVITIES",
    "J": "INFORMATION AND COMMUNICATION",
    "K": "FINANCIAL AND INSURANCE ACTIVITIES",
    "L": "REAL ESTATE ACTIVITIES",
    "M": "PROFESSIONAL, SCIENTIFIC AND TECHNICAL ACTIVITIES",
    "N": "ADMINISTRATIVE AND SUPPORT SERVICE ACTIVITIES",
    "O": "PUBLIC ADMINISTRATION AND DEFENCE; COMPULSORY SOCIAL SECURITY",
    "P": "EDUCATION",
    "Q": "HUMAN HEALTH AND SOCIAL WORK ACTIVITIES",
    "R": "ARTS, ENTERTAINMENT AND RECREATION",
    "S": "OTHER SERVICE ACTIVITIES",
    "T": "ACTIVITIES OF HOUSEHOLDS AS EMPLOYERS; UNDIFFERENTIATED GOODS- AND SERVICES-PRODUCING ACTIVITIES OF HOUSEHOLDS FOR OWN USE",
    "U": "ACTIVITIES OF EXTRATERRITORIAL ORGANISATIONS AND BODIES"
}


def parse_sector_metadata(raw_sector: str):
    """Parse a raw sector label into structured sector metadata."""
    if not raw_sector or not isinstance(raw_sector, str):
        return None

    raw_sector = raw_sector.strip()

    # Split description
    parts = raw_sector.split(" - ")
    code_part = parts[0].strip()
    description = parts[1].strip() if len(parts) > 1 else None

    # Extract the section letter
    section_code = code_part[0] if code_part else None
    section_name = NACE_SECTIONS.get(section_code, None)

    # Extract numeric parts (e.g., 21, 20)
    numeric_parts = re.findall(r"\d+", code_part)
    division = numeric_parts[0] if len(numeric_parts) >= 1 else None
    group = numeric_parts[1][0] if len(numeric_parts) >= 2 else None
    class_code = numeric_parts[1][1:] if len(numeric_parts) >= 2 and len(numeric_parts[1]) > 1 else None

    return {
        "sectionCode": section_code,
        "sectionName": section_name,
        "division": division,
        "group": group,
        "classCode": class_code,
        "classDescription": description
    }


# === Transform in memory (filtered_cases) ===
for case_id, case_data in filtered_cases.items():
    raw_sector = case_data.get("caseSectorLabel")
    case_data["sector_metadata"] = parse_sector_metadata(raw_sector)

print(f"✅ Sector metadata parsed for {len(filtered_cases)} cases (in memory).")

✅ Sector metadata parsed for 10919 cases (in memory).


# Clean company names (in memory, before CSV)

Drops boilerplate entries (`JV`, `KKR`, `CVC`) from each case's `caseCompanies`
list. Moved ahead of the CSV step so it works on the list directly instead of
re-splitting a joined string out of `cases.csv`.

In [17]:
# --- Clean company names in memory, BEFORE the CSV conversion ---
COMPANIES_TO_REMOVE = {"JV", "KKR", "CVC"}

removed = 0
for case_data in filtered_cases.values():
    comps = case_data.get("caseCompanies", [])
    if isinstance(comps, str):  # defensive: split legacy "/"-joined string
        comps = [c.strip() for c in comps.split("/")]
    cleaned = [
        c.strip() for c in comps
        if isinstance(c, str) and c.strip()
        and c.strip().upper() not in COMPANIES_TO_REMOVE
    ]
    removed += len(comps) - len(cleaned)
    case_data["caseCompanies"] = cleaned

print(f"✅ Removed {removed} blacklisted company entries (in memory).")

✅ Removed 1337 blacklisted company entries (in memory).


# Converting the Json to CSV

In [18]:
import pandas as pd
import numpy as np

records = []
for case_id, case_data in filtered_cases.items():
    # Get the nested metadata safely
    sector_meta = case_data.get("sector_metadata") or {}

    record = {
        "caseId": case_id,
        "caseInstrument": case_data.get("caseInstrument") or None,
        "caseNumber": case_data.get("caseNumber") or None,
        "caseTitle": case_data.get("caseTitle") or None,
        "caseSectorLabel": case_data.get("caseSectorLabel") or None,
        "caseLastDecisionDate": case_data.get("caseLastDecisionDate") or None,
        "caseInitiationDate": case_data.get("caseInitiationDate") or None,
        "decisionLabel": case_data.get("decisionLabel") or None,
        "sectionCode": sector_meta.get("sectionCode") or None,
        "sectionName": sector_meta.get("sectionName") or None,
        "division": sector_meta.get("division") or None,
        "group": sector_meta.get("group") or None,
        "classCode": sector_meta.get("classCode") or None,
        "classDescription": sector_meta.get("classDescription") or None,
        "caseCompanies": "; ".join(case_data.get("caseCompanies", [])) or None,
        "caseLegalBasisLabel": "; ".join(case_data.get("caseLegalBasisLabel", [])) or None,
    }

    records.append(record)

df = pd.DataFrame(records)

# Normalize blanks / whitespace / literal "nan" -> NaN
# (previously a separate read-back pass over cases.csv)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.replace({"": np.nan, "NaN": np.nan, "nan": np.nan})

# Single write, missing values rendered as "null"
df.to_csv("cases.csv", index=False, na_rep="null", encoding="utf-8")
print(f"✅ Wrote {len(df)} rows to cases.csv")

✅ Wrote 10919 rows to cases.csv


In [20]:
pd.set_option('display.max_columns', None)


In [23]:
filtered_cases

{'M.2027': {'caseInstrument': 'Merger',
  'caseNumber': '2027',
  'caseTitle': 'DEUTSCHE BANK / SAP / JV',
  'caseSectorsCode': 'NaceSectorsG_46',
  'caseSectorLabel': 'G.46 - Wholesale trade, except of motor vehicles and motorcycles',
  'caseCompanies': ['DEUTSCHE BANK', 'SAP'],
  'caseLegalBasisCode': '',
  'caseLegalBasisLabel': ['Art. 105'],
  'caseLastDecisionDate': '2000-07-13',
  'caseInitiationDate': '2000-06-09',
  'decisions': [{'decisionDate': '2000-07-13',
    'decisionLabel': 'Art. 6(1)(b)'},
   {'decisionDate': '', 'decisionLabel': ''}],
  'sector_metadata': {'sectionCode': 'G',
   'sectionName': 'WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VEHICLES AND MOTORCYCLES',
   'division': '46',
   'group': None,
   'classCode': None,
   'classDescription': 'Wholesale trade, except of motor vehicles and motorcycles'}},
 'M.2028': {'caseInstrument': 'Merger',
  'caseNumber': '2028',
  'caseTitle': 'ABB / BILFINGER / MVV ENERGIE / JV',
  'caseSectorsCode': 'NaceSectorsE_36_0_0',
  

In [21]:
df

,caseId,caseInstrument,caseNumber,caseTitle,caseSectorLabel,caseLastDecisionDate,caseInitiationDate,decisionLabel,sectionCode,sectionName,division,group,classCode,classDescription,caseCompanies,caseLegalBasisLabel
0,M.2027,Merger,2027,DEUTSCHE BANK / SAP / JV,"G.46 - Wholesale trade, except of motor vehicl...",2000-07-13,2000-06-09,None,G,WHOLESALE AND RETAIL TRADE; REPAIR OF MOTOR VE...,46,None,None,"Wholesale trade, except of motor vehicles and ...",DEUTSCHE BANK; SAP,Art. 105
1,M.2028,Merger,2028,ABB / BILFINGER / MVV ENERGIE / JV,"E.36.00 - Water collection, treatment and supply",2000-07-25,2000-06-22,None,E,"WATER SUPPLY; SEWERAGE, WASTE MANAGEMENT AND R...",36,0,0,"Water collection, treatment and supply",ABB; BILFINGER; MVV ENERGIE,Art. 105
2,M.422,Merger,422,UNILEVER FRANCE / ORTIZ MIKO (II),C.10.52 - Manufacture of ice cream,1994-03-15,1994-02-14,None,C,MANUFACTURING,10,5,2,Manufacture of ice cream,UNILEVER FRANCE; ORTIZ MIKO (II),Art. 105
3,M.2029,Merger,2029,TATE & LYLE / AMYLUM,C.10.62 - Manufacture of starches and starch p...,2000-08-11,2000-07-07,None,C,MANUFACTURING,10,6,2,Manufacture of starches and starch products,TATE & LYLE; AMYLUM,Art. 105
4,M.420,Merger,420,KPR / CGP,C.28.22 - Manufacture of lifting and handling ...,1994-04-14,1994-03-09,None,C,MANUFACTURING,28,2,2,Manufacture of lifting and handling equipment,KPR; CGP,Art. 105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10914,AT.39942,Antitrust & Cartels,39942,Ryanair/Malaga Airport,H.51.10 - Passenger air transport,2015-02-25,2011-10-21,None,H,TRANSPORTATION AND STORAGE,51,1,0,Passenger air transport,None,Art. 106; Art. 102
10915,AT.39943,Antitrust & Cartels,39943,E5 - Cooperation among large telecom operators,J.61 - Telecommunications,None,2011-10-27,None,J,INFORMATION AND COMMUNICATION,61,None,None,Telecommunications,None,Art. 101
10916,AT.39822,Antitrust & Cartels,39822,Refrigerants,C.29.32 - Manufacture of other parts and acces...,2017-10-25,2010-06-29,None,C,MANUFACTURING,29,3,2,Manufacture of other parts and accessories for...,None,Art. 101; Art. 102
10917,AT.39824,Antitrust & Cartels,39824,Trucks,C.29.10 - Manufacture of motor vehicles,2017-09-27,2010-08-02,None,C,MANUFACTURING,29,1,0,Manufacture of motor vehicles,Renault Trucks SAS; MAN SE; Iveco Magirus AG; ...,Art. 101
